In [1]:
import subprocess

def call_claude(prompt, system=None):
    cmd = ["claude", "-p", prompt]
    if system:
        cmd += ["--system-prompt", system]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return result.stdout.strip()


In [2]:
# Helper functions
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    prompt = "\n\n".join(
        f"{'User' if m['role'] == 'user' else 'Assistant'}: {m['content']}"
        for m in messages
    )
    return call_claude(prompt, system=system)


In [4]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        \"task\": \"Description of task\",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    response = chat(messages)

    # Extract JSON from markdown code block if present
    if "```json" in response:
        response = response.split("```json")[1].split("```")[0].strip()
    elif "```" in response:
        response = response.split("```")[1].split("```")[0].strip()

    return json.loads(response)


In [6]:
dataset = generate_dataset()
print(json.dumps(dataset, indent=2))
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

[
  {
    "task": "Write a Python function that takes an S3 bucket name and prefix, and returns a list of all object keys using boto3's paginator to handle buckets with more than 1000 objects."
  },
  {
    "task": "Write a JSON IAM policy document that allows a Lambda function to read objects from a specific S3 bucket and write logs to CloudWatch Logs, following least-privilege principles."
  },
  {
    "task": "Write a regular expression that validates an AWS ARN format, matching strings like 'arn:aws:s3:::my-bucket' or 'arn:aws:iam::123456789012:role/MyRole' while rejecting malformed ARNs."
  }
]


In [8]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:
    {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [9]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [10]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [11]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)

In [12]:
print(json.dumps(results, indent=2))

[
  {
    "output": "```python\nimport boto3\n\n\ndef list_s3_objects(bucket_name: str, prefix: str = \"\") -> list[str]:\n    s3 = boto3.client(\"s3\")\n    paginator = s3.get_paginator(\"list_objects_v2\")\n    \n    keys = []\n    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):\n        for obj in page.get(\"Contents\", []):\n            keys.append(obj[\"Key\"])\n    \n    return keys\n```\n\nKey points:\n- `get_paginator(\"list_objects_v2\")` automatically handles the 1000-object limit by issuing multiple requests and yielding each page.\n- `page.get(\"Contents\", [])` guards against empty pages (e.g., when no objects match the prefix).\n- Pass `prefix=\"\"` to list the entire bucket, or a specific prefix like `\"logs/2024/\"` to scope the results.",
    "test_case": {
      "task": "Write a Python function that takes an S3 bucket name and prefix, and returns a list of all object keys using boto3's paginator to handle buckets with more than 1000 objects."
    },